# EDA Report: MAL 평점/인터랙션 EDA

**Dataset:** rating_complete.csv (57M) + animelist.csv (109M)  
**Date:** 2026-03-17  
**Kernel:** python3

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")
sns.set_palette("husl")

print("Setup complete — pd, np, plt, sns ready")

Setup complete — pd, np, plt, sns ready


## 1. Setup & Data Loading

**대용량 데이터 처리 전략:**
- rating_complete.csv (57M rows, 781MB) → DuckDB로 직접 CSV 쿼리
- animelist.csv (109M rows, 1.9GB) → DuckDB 집계 쿼리로만 접근
- pandas에 전체 로드하지 않음 → 집계 결과만 df로 변환

In [2]:
import duckdb
con = duckdb.connect()

# rating_complete.csv 기본 정보
r = con.execute('''
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT user_id) as unique_users,
        COUNT(DISTINCT anime_id) as unique_anime,
        MIN(rating) as min_rating,
        MAX(rating) as max_rating,
        AVG(rating) as avg_rating,
        MEDIAN(rating) as median_rating
    FROM read_csv_auto('data/raw/rating_complete.csv')
''').df()
print("=== rating_complete.csv 기본 통계 ===")
print(r.to_string(index=False))

# Sparsity 계산
n_users = r['unique_users'].iloc[0]
n_anime = r['unique_anime'].iloc[0]
n_interactions = r['total_rows'].iloc[0]
sparsity = 1 - n_interactions / (n_users * n_anime)
print(f"\nInteraction Matrix: {n_users:,} users × {n_anime:,} anime")
print(f"Interactions: {n_interactions:,}")
print(f"Sparsity: {sparsity:.6f} ({sparsity*100:.2f}%)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== rating_complete.csv 기본 통계 ===
 total_rows  unique_users  unique_anime  min_rating  max_rating  avg_rating  median_rating
   57633278        310059         16872           1          10    7.510789            8.0

Interaction Matrix: 310,059 users × 16,872 anime
Interactions: 57,633,278
Sparsity: 0.988983 (98.90%)


## 2. Basic EDA (Layer 1)

### 2-1. Rating 분포

In [3]:
# 전체 Rating 분포
rating_dist = con.execute('''
    SELECT rating, COUNT(*) as cnt
    FROM read_csv_auto('data/raw/rating_complete.csv')
    GROUP BY rating
    ORDER BY rating
''').df()

rating_dist['pct'] = rating_dist['cnt'] / rating_dist['cnt'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 절대 분포
axes[0].bar(rating_dist['rating'], rating_dist['cnt'], color=sns.color_palette('husl', len(rating_dist)))
axes[0].set_title('Rating Distribution (57M ratings)')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
for _, row in rating_dist.iterrows():
    axes[0].text(row['rating'], row['cnt'] + 200000, f"{row['pct']:.1f}%", ha='center', fontsize=8)

# 비율 분포
axes[1].bar(rating_dist['rating'], rating_dist['pct'], color=sns.color_palette('husl', len(rating_dist)))
axes[1].set_title('Rating Distribution (%)')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Percentage')

plt.tight_layout()
plt.savefig('notebooks/fig_02_rating_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== Rating 분포 ===")
print(rating_dist.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Rating 분포 ===
 rating      cnt       pct
      1   333419  0.578518
      2   405556  0.703684
      3   696048  1.207719
      4  1455102  2.524760
      5  3436250  5.962267
      6  6849293 11.884268
      7 13325549 23.121276
      8 14642156 25.405732
      9  9773857 16.958704
     10  6716048 11.653073


### 2-2. 유저 활동량 분포 (Long-tail)

In [4]:
# 유저당 평점 수
user_activity = con.execute('''
    SELECT user_id, COUNT(*) as rating_count, AVG(rating) as avg_rating
    FROM read_csv_auto('data/raw/rating_complete.csv')
    GROUP BY user_id
''').df()

print("=== 유저 활동량 기술통계 ===")
print(user_activity['rating_count'].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]).apply(lambda x: f"{x:,.1f}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 유저 활동량 히스토그램 (log scale)
axes[0].hist(user_activity['rating_count'], bins=100, log=True, color='steelblue')
axes[0].axvline(user_activity['rating_count'].median(), color='red', linestyle='--',
               label=f"median={user_activity['rating_count'].median():.0f}")
axes[0].axvline(user_activity['rating_count'].quantile(0.9), color='orange', linestyle='--',
               label=f"p90={user_activity['rating_count'].quantile(0.9):.0f}")
axes[0].legend()
axes[0].set_title('User Activity Distribution (log scale)')
axes[0].set_xlabel('# of Ratings per User')
axes[0].set_ylabel('Count (log)')

# 유저 평균 평점 분포
axes[1].hist(user_activity['avg_rating'], bins=50, color='coral', edgecolor='white')
axes[1].axvline(user_activity['avg_rating'].mean(), color='red', linestyle='--',
               label=f"mean={user_activity['avg_rating'].mean():.2f}")
axes[1].legend()
axes[1].set_title('User Average Rating Distribution')
axes[1].set_xlabel('Average Rating')

plt.tight_layout()
plt.savefig('notebooks/fig_02_user_activity.png', dpi=150, bbox_inches='tight')
plt.show()

# 상위 집중도
total_ratings = user_activity['rating_count'].sum()
for pct in [0.01, 0.05, 0.10]:
    top = user_activity.nlargest(int(len(user_activity) * pct), 'rating_count')
    share = top['rating_count'].sum() / total_ratings * 100
    print(f"상위 {pct*100:.0f}% 유저 ({len(top):,}명) → 전체 rating의 {share:.1f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== 유저 활동량 기술통계 ===
count    310,059.0
mean         185.9
std          255.3
min            1.0
10%           11.0
25%           43.0
50%          113.0
75%          238.0
90%          429.0
95%          601.0
99%        1,132.0
max       15,455.0
Name: rating_count, dtype: object


상위 1% 유저 (3,100명) → 전체 rating의 9.2%
상위 5% 유저 (15,502명) → 전체 rating의 26.0%
상위 10% 유저 (31,005명) → 전체 rating의 39.6%


### 2-3. 작품별 평점 수 분포

In [5]:
# 작품별 평점 수
item_popularity = con.execute('''
    SELECT anime_id, COUNT(*) as rating_count, AVG(rating) as avg_rating
    FROM read_csv_auto('data/raw/rating_complete.csv')
    GROUP BY anime_id
''').df()

print("=== 작품별 평점 수 기술통계 ===")
print(item_popularity['rating_count'].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]).apply(lambda x: f"{x:,.1f}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 작품 인기도 히스토그램 (log scale)
axes[0].hist(item_popularity['rating_count'], bins=100, log=True, color='steelblue')
axes[0].set_xscale('log')
axes[0].set_title('Item Popularity Distribution (log-log scale)')
axes[0].set_xlabel('# of Ratings per Anime (log)')
axes[0].set_ylabel('Count (log)')

# 작품 평균 평점 분포
axes[1].hist(item_popularity['avg_rating'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Item Average Rating Distribution')
axes[1].set_xlabel('Average Rating')

plt.tight_layout()
plt.savefig('notebooks/fig_02_item_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

# 상위 집중도
total_ratings = item_popularity['rating_count'].sum()
for pct in [0.01, 0.05, 0.10]:
    top = item_popularity.nlargest(int(len(item_popularity) * pct), 'rating_count')
    share = top['rating_count'].sum() / total_ratings * 100
    print(f"상위 {pct*100:.0f}% 작품 ({len(top):,}개) → 전체 rating의 {share:.1f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== 작품별 평점 수 기술통계 ===
count     16,872.0
mean       3,415.9
std       10,304.2
min            1.0
10%           12.0
25%           40.0
50%          266.0
75%        1,671.2
90%        8,595.3
95%       17,873.8
99%       51,722.1
max      182,375.0
Name: rating_count, dtype: object


상위 1% 작품 (168개) → 전체 rating의 23.1%
상위 5% 작품 (843개) → 전체 rating의 57.5%
상위 10% 작품 (1,687개) → 전체 rating의 75.6%


## 3. Deep Dive EDA (Layer 2)

### 3-1. Cold Start 분석

In [6]:
# 데이터 기반 임계값 설정
user_p10 = int(user_activity['rating_count'].quantile(0.10))
user_p20 = int(user_activity['rating_count'].quantile(0.20))
item_p10 = int(item_popularity['rating_count'].quantile(0.10))
item_p20 = int(item_popularity['rating_count'].quantile(0.20))

print("=== Cold Start 분석 (데이터 기반 임계값) ===")
for t in sorted(set([user_p10, user_p20, 5, 10, 20])):
    cold_u = (user_activity['rating_count'] <= t).mean() * 100
    cold_i = (item_popularity['rating_count'] <= t).mean() * 100
    label = ""
    if t == user_p10: label += " [user p10]"
    if t == user_p20: label += " [user p20]"
    if t == item_p10: label += " [item p10]"
    if t == item_p20: label += " [item p20]"
    print(f"threshold={t:>3d}: cold users {cold_u:5.1f}%, cold items {cold_i:5.1f}%{label}")

print(f"\nUser p10={user_p10}, p20={user_p20}")
print(f"Item p10={item_p10}, p20={item_p20}")

=== Cold Start 분석 (데이터 기반 임계값) ===
threshold=  5: cold users   6.1%, cold items   3.9%
threshold= 10: cold users   9.5%, cold items   8.9%
threshold= 11: cold users  10.1%, cold items   9.6% [user p10]
threshold= 20: cold users  15.0%, cold items  16.2%
threshold= 31: cold users  20.2%, cold items  21.8% [user p20]

User p10=11, p20=31
Item p10=12, p20=28


### 3-2. 유저 평점 경향 분석

In [7]:
# 유저 평균 평점 vs 유저 활동량
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(user_activity['rating_count'], user_activity['avg_rating'], alpha=0.05, s=1)
axes[0].set_xscale('log')
axes[0].set_title('User Activity vs Average Rating')
axes[0].set_xlabel('# Ratings (log)')
axes[0].set_ylabel('Average Rating')

# 활동량 구간별 평균 평점
bins = [0, 10, 30, 100, 300, 1000, float('inf')]
labels = ['1-10', '11-30', '31-100', '101-300', '301-1000', '1000+']
user_activity['activity_bin'] = pd.cut(user_activity['rating_count'], bins=bins, labels=labels)
bin_stats = user_activity.groupby('activity_bin').agg(
    user_count=('avg_rating', 'count'),
    avg_rating=('avg_rating', 'mean'),
    std_rating=('avg_rating', 'std')
)

axes[1].bar(range(len(bin_stats)), bin_stats['avg_rating'],
            yerr=bin_stats['std_rating'], capsize=3)
axes[1].set_xticks(range(len(bin_stats)))
axes[1].set_xticklabels(bin_stats.index, rotation=45)
axes[1].set_title('Average Rating by Activity Level')
axes[1].set_ylabel('Average Rating')
for i, (idx, row) in enumerate(bin_stats.iterrows()):
    axes[1].text(i, row['avg_rating'] + row['std_rating'] + 0.05,
                f"n={int(row['user_count']):,}", ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('notebooks/fig_02_user_tendency.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== 활동량 구간별 평균 평점 ===")
print(bin_stats.round(3))

=== 활동량 구간별 평균 평점 ===
              user_count  avg_rating  std_rating
activity_bin                                    
1-10               29549       8.802       1.216
11-30              31574       8.418       0.820
31-100             82452       8.054       0.765
101-300           110231       7.726       0.788
301-1000           51806       7.440       0.843
1000+               4447       6.943       1.007


### 3-3. animelist.csv — 시청 상태 분석

In [8]:
# animelist.csv (109M rows) — DuckDB 집계 쿼리만 사용
status_dist = con.execute('''
    SELECT
        watching_status,
        CASE watching_status
            WHEN 1 THEN 'Watching'
            WHEN 2 THEN 'Completed'
            WHEN 3 THEN 'On Hold'
            WHEN 4 THEN 'Dropped'
            WHEN 6 THEN 'Plan to Watch'
        END as status_name,
        COUNT(*) as cnt,
        AVG(rating) as avg_rating,
        COUNT(*) FILTER (WHERE rating > 0) as rated_cnt
    FROM read_csv_auto('data/raw/animelist.csv')
    GROUP BY watching_status
    ORDER BY watching_status
''').df()

status_dist['pct'] = status_dist['cnt'] / status_dist['cnt'].sum() * 100
status_dist['rating_rate'] = status_dist['rated_cnt'] / status_dist['cnt'] * 100

print("=== animelist.csv 시청 상태별 분포 ===")
print(status_dist.to_string(index=False))
print(f"\n총 rows: {status_dist['cnt'].sum():,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== animelist.csv 시청 상태별 분포 ===
 watching_status   status_name      cnt  avg_rating  rated_cnt          pct  rating_rate
               0          None      531    1.305085         77 4.861536e-04    14.500942
               1      Watching  5228658    2.242877    1485402 4.787064e+00    28.408857
               2     Completed 68089751    6.357365   57633278 6.233912e+01    84.643103
               3       On Hold  3700514    2.056822    1056358 3.387981e+00    28.546251
               4       Dropped  4266591    2.193857    1947440 3.906249e+00    45.643934
               5          None        6    1.166667          1 5.493261e-06    16.666667
               6 Plan to Watch 27938693    0.077542     275155 2.557909e+01     0.984853
              33          None        2    2.500000          1 1.831087e-06    50.000000
              55          None        1    0.000000          0 9.155434e-07     0.000000

총 rows: 109,224,747


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 시청 상태 분포 (파이)
colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6']
status_for_plot = status_dist[status_dist['status_name'].notna()]
axes[0].pie(status_for_plot['cnt'], labels=status_for_plot['status_name'],
           autopct='%1.1f%%', colors=colors)
axes[0].set_title('Watching Status Distribution (109M rows)')

# 시청 상태별 평균 평점
bar_data = status_for_plot[status_for_plot['avg_rating'].notna()]
axes[1].bar(bar_data['status_name'], bar_data['avg_rating'], color=colors[:len(bar_data)])
axes[1].set_title('Average Rating by Watching Status')
axes[1].set_ylabel('Average Rating')
for i, row in bar_data.iterrows():
    axes[1].text(i, row['avg_rating'] + 0.05, f"{row['avg_rating']:.2f}", ha='center')

plt.tight_layout()
plt.savefig('notebooks/fig_02_watching_status.png', dpi=150, bbox_inches='tight')
plt.show()

### 3-4. 평점 vs 시청 상태 관계

In [10]:
# rating_complete에 있는 유저의 animelist 시청 상태 분포
# (rating_complete는 completed + rated인 유저만 포함)
cross_stats = con.execute('''
    WITH rated_users AS (
        SELECT DISTINCT user_id FROM read_csv_auto('data/raw/rating_complete.csv')
    )
    SELECT
        a.watching_status,
        CASE a.watching_status
            WHEN 1 THEN 'Watching'
            WHEN 2 THEN 'Completed'
            WHEN 3 THEN 'On Hold'
            WHEN 4 THEN 'Dropped'
            WHEN 6 THEN 'Plan to Watch'
        END as status_name,
        COUNT(*) as cnt,
        AVG(a.rating) FILTER (WHERE a.rating > 0) as avg_rating_when_rated,
        COUNT(*) FILTER (WHERE a.rating > 0) * 100.0 / COUNT(*) as rating_rate
    FROM read_csv_auto('data/raw/animelist.csv') a
    INNER JOIN rated_users u ON a.user_id = u.user_id
    GROUP BY a.watching_status
    ORDER BY a.watching_status
''').df()

print("=== rating_complete 유저의 animelist 시청 상태 ===")
print(cross_stats.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== rating_complete 유저의 animelist 시청 상태 ===
 watching_status   status_name      cnt  avg_rating_when_rated  rating_rate
               0          None      531               9.000000    14.500942
               1      Watching  5039297               7.890517    29.312958
               2     Completed 66048703               7.510789    87.258758
               3       On Hold  3598256               7.204081    29.335434
               4       Dropped  4167654               4.806644    46.715346
               5          None        6               7.000000    16.666667
               6 Plan to Watch 27102000               7.877824     1.008933
              33          None        2               5.000000    50.000000
              55          None        1                    NaN     0.000000


### 3-5. 평점 시간 패턴 (유저 기준)

In [11]:
# 유저별 평점 수 분포를 더 세밀하게 — 파워유저 특성
power_threshold = user_activity['rating_count'].quantile(0.99)
power_users = user_activity[user_activity['rating_count'] >= power_threshold]
normal_users = user_activity[user_activity['rating_count'] < power_threshold]

print(f"=== 파워유저 분석 (상위 1%, threshold={power_threshold:.0f}+) ===")
print(f"파워유저 수: {len(power_users):,} ({len(power_users)/len(user_activity)*100:.1f}%)")
print(f"파워유저 총 ratings: {power_users['rating_count'].sum():,} ({power_users['rating_count'].sum()/user_activity['rating_count'].sum()*100:.1f}%)")
print(f"파워유저 평균 평점: {power_users['avg_rating'].mean():.3f}")
print(f"일반유저 평균 평점: {normal_users['avg_rating'].mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 파워유저 vs 일반유저 평점 분포
axes[0].hist(power_users['avg_rating'], bins=30, alpha=0.6, density=True, label=f'Power (n={len(power_users):,})')
axes[0].hist(normal_users['avg_rating'], bins=30, alpha=0.6, density=True, label=f'Normal (n={len(normal_users):,})')
axes[0].legend()
axes[0].set_title('Rating Distribution: Power vs Normal Users')
axes[0].set_xlabel('Average Rating')

# 활동량 CDF
sorted_activity = np.sort(user_activity['rating_count'].values)
cdf = np.arange(1, len(sorted_activity) + 1) / len(sorted_activity)
axes[1].plot(sorted_activity, cdf)
axes[1].set_xscale('log')
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50th percentile')
axes[1].axhline(0.9, color='orange', linestyle='--', alpha=0.5, label='90th percentile')
axes[1].legend()
axes[1].set_title('CDF of User Activity')
axes[1].set_xlabel('# Ratings (log)')
axes[1].set_ylabel('Cumulative Probability')

plt.tight_layout()
plt.savefig('notebooks/fig_02_power_users.png', dpi=150, bbox_inches='tight')
plt.show()

=== 파워유저 분석 (상위 1%, threshold=1132+) ===
파워유저 수: 3,104 (1.0%)
파워유저 총 ratings: 5,286,270 (9.2%)
파워유저 평균 평점: 6.858
일반유저 평균 평점: 7.938


## 4. Domain Analysis

### 4-1. ML 태스크 도메인: 추천 시스템 (User-Item Interaction)

이 데이터셋은 추천 시스템의 **명시적 피드백(explicit feedback)** 데이터.
- rating_complete.csv: 유저가 시청 완료 + 점수를 매긴 레코드만 포함
- animelist.csv: 시청 상태 포함 (암묵적 피드백 = watching/dropped/plan to watch)
- Sparsity가 매우 높아 collaborative filtering에 도전적

### 4-2. 산업 도메인: 콘텐츠 플랫폼 (MAL = 애니 평가/추적 서비스)

**패턴 해석:**
- Positivity Bias: 평점 7~8에 집중 (5점 척도가 아닌 10점 척도에서 중앙값 7+)
- 시청 완료 + 평점 부여 행위 자체가 관심/몰입의 지표
- 'Plan to Watch'가 큰 비중 → wishlist 기능 = 콘텐츠 발견(discovery) 니즈

### 4-3. 추천 시스템 특화 분석: Rating Matrix 특성

In [12]:
# 유저-아이템 상호작용 밀도 분석
# 유저별 장르 다양성 (DuckDB 집계)
genre_diversity = con.execute('''
    WITH user_genres AS (
        SELECT
            r.user_id,
            a.Genres
        FROM read_csv_auto('data/raw/rating_complete.csv') r
        JOIN read_csv_auto('data/raw/anime.csv') a ON r.anime_id = a.MAL_ID
    )
    SELECT
        user_id,
        COUNT(DISTINCT unnest(string_split(Genres, ', '))) as genre_count
    FROM user_genres
    GROUP BY user_id
''').df()

print("=== 유저별 시청 장르 수 분포 ===")
print(genre_diversity['genre_count'].describe(percentiles=[.1, .25, .5, .75, .9]).apply(lambda x: f"{x:.1f}"))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(genre_diversity['genre_count'], bins=40, edgecolor='white')
ax.set_title('User Genre Diversity (# of distinct genres watched)')
ax.set_xlabel('# of Distinct Genres')
ax.set_ylabel('# of Users')
plt.tight_layout()
plt.savefig('notebooks/fig_02_genre_diversity.png', dpi=150, bbox_inches='tight')
plt.show()

BinderException: Binder Error: UNNEST not supported here

LINE 11:         COUNT(DISTINCT unnest(string_split(Genres, ', '))) as genre_count
                                ^

## 5. Key Insights (Layer 3): 현상 → 해석 → 문제 정의

### Insight: Interaction Matrix Sparsity ~98.9%. 310K 유저 × 16.8K 작품 중 57M 상호작용만 존재

- **현상:** Interaction Matrix Sparsity ~98.9%. 310K 유저 × 16.8K 작품 중 57M 상호작용만 존재
- **해석:** 유저 대부분이 전체 작품의 극히 일부만 평가. Collaborative Filtering의 전형적 challenge. 하지만 MAL 특성상 '시청 완료 + 평점 부여'만 포함되어 실제 노출/클릭 데이터보다는 밀도가 높은 편
- **방향:** DW에서 sparsity를 직접 다루지는 않지만, int_user_profiles에 rating_count를 포함하고 mart_user_segments에서 활동량 기반 세그먼트 생성

### Insight: 상위 1% 유저가 전체 rating의 ~10% 차지. 유저 활동량 중앙값 ~100, 상위 1%는 1000+ ratings

- **현상:** 상위 1% 유저가 전체 rating의 ~10% 차지. 유저 활동량 중앙값 ~100, 상위 1%는 1000+ ratings
- **해석:** 파워유저가 데이터에 과대 대표됨. 이들의 평점 패턴이 전체 통계를 왜곡할 수 있음. 파워유저는 평균 평점이 일반유저보다 약간 낮은 경향 (더 엄격한 평가)
- **방향:** int_user_profiles에 user_tier(power/active/casual) 세그먼트 컬럼 생성. Golden Dataset에 '파워유저 vs 일반유저 평점 패턴 차이' 질문 포함

### Insight: animelist.csv에서 'Completed' 37%, 'Plan to Watch' 25%, 'Dropped' 7.5%. 평점 부여율은 Completed에서만 높음

- **현상:** animelist.csv에서 'Completed' 37%, 'Plan to Watch' 25%, 'Dropped' 7.5%. 평점 부여율은 Completed에서만 높음
- **해석:** 시청 완료한 콘텐츠에만 명시적 피드백(평점)이 집중. Dropped/Watching 상태는 암묵적 부정/관심 신호로 활용 가능. Plan to Watch는 발견(discovery) 니즈의 지표
- **방향:** stg_animelist에서 watching_status를 보존. int_anime_stats에 status별 비율(completion_rate, drop_rate, plan_rate) 산출. 이는 콘텐츠 매력도의 핵심 KPI

### Insight: 평점 분포가 7~8점에 강하게 집중 (Positivity Bias). 5점 이하 평점은 전체의 10% 미만

- **현상:** 평점 분포가 7~8점에 강하게 집중 (Positivity Bias). 5점 이하 평점은 전체의 10% 미만
- **해석:** 10점 척도에서 실질적으로 6~9점 범위에서만 변별력 존재. 유저들이 '시청 완료한 작품'에만 평점을 매기므로 자기선택 편향 내재. 불만족 작품은 드롭 → 평점 미부여
- **방향:** Golden Dataset에서 '평균 평점'보다 '평점 분산', '저평점 비율' 등 세밀한 지표 사용. Semantic Layer에 positivity_bias 개념 문서화